# 05 — Week 4: the strength/depth confound, and what the matcher could see

*Written 23 August 2026. Read `docs/HANDOVER.md` first.*

This notebook does three jobs, in order of how much they matter:

1. **§2 — why layer 16 was ever chosen.** Week 1 selected the operating point by
   `max(sweep, key=sweep.get)` over `refusal_rate`, the substring matcher. This section shows the
   matcher sits at **exactly 100% for layers 10–18** and 97.9% at layer 20, while underneath it the
   true clean-refusal rate runs 93.8% → 97.9% → 47.9% → 12.5% → 10.4%. The selection criterion is
   flat across an 85-point swing in the thing it was standing in for. CPU-only — it re-scores
   generations already on disk.
2. **§3–§5 — reproduce the 22 August layer sweep.** `steps123_results.json` is the evidence for the
   reversal and **no code in the repository produced it**; it was run in a session that was never
   saved. These sections regenerate it, and §3.2 diffs the new run against the stored one.
3. **§6–§7 — settle the confound.** `multiplier` scales the *raw* vector and `||v||` grows 6.2x
   with depth, so every layer comparison this project has made varied depth and strength together.
   §6 is the pre-registered matched-norm test; §7 puts both on a dimensionless relative axis.

**Runs unchanged locally or on Colab.** The bootstrap cell below detects which, installs the package
if needed, and resolves every path from the repo root — so it does not matter where the kernel was
started. §2 and §3.2 are CPU-only; §3.1 and §4–§7 need a GPU and defer cleanly without one.

## §0 Setup

In [ ]:
# %% 0.0 BOOTSTRAP -- run this first, always. Identical locally and on Colab.
import os, subprocess, sys
from pathlib import Path

# Colab only. The repo is committed WITHOUT a token: put a fine-grained GitHub PAT with read
# access in Colab Secrets (the key icon in the sidebar) under this name, so no credential is
# ever pasted into a notebook that gets committed.
GITHUB_REPO   = "YOUR-USERNAME/adass"      # <- set once
COLAB_SECRET  = "GH_TOKEN"
DRIVE_DIR     = "/content/drive/MyDrive/adass"   # fallback if you'd rather not use GitHub

IN_COLAB = "google.colab" in sys.modules


def _find_root(start):
    """Walk up looking for the repo: a pyproject.toml sitting next to the adass package."""
    p = Path(start).resolve()
    for cand in (p, *p.parents):
        if (cand / "pyproject.toml").is_file() and (cand / "adass" / "core.py").is_file():
            return cand
    return None


ROOT = _find_root(Path.cwd())

if IN_COLAB and ROOT is None:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get(COLAB_SECRET)
    except Exception:
        pass
    if token:
        url = f"https://{token}@github.com/{GITHUB_REPO}.git"
        subprocess.run(["git", "clone", "--quiet", url, "/content/adass"], check=True)
        ROOT = Path("/content/adass")
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = _find_root(DRIVE_DIR) or Path(DRIVE_DIR)

assert ROOT is not None, "repo not found -- set GITHUB_REPO + the Colab secret, or DRIVE_DIR"
os.chdir(ROOT)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
    # Both source datasets are GATED: the token needs google/gemma-2-2b-it AND walledai/AdvBench
    # accepted. Note we do NOT set HF_HUB_OFFLINE here -- nothing is cached on a fresh runtime.
    from huggingface_hub import get_token, notebook_login
    if not (get_token() or os.environ.get("HF_TOKEN")):
        notebook_login()
elif str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import adass
print(adass.paths.describe())

In [ ]:
# %% 0.1 Run flags and the one deliberate truncation point.
import json, math
from collections import Counter

OUT = "week4_layers.json"          # adass.save_results resolves bare names to data/results/

# §2 and §3.2 are CPU-only (they re-score text already on disk). §3.1, §4-§7 need the model.
LOAD_MODEL = os.environ.get("ADASS_LOAD_MODEL", "0") == "1"
print(f"LOAD_MODEL={LOAD_MODEL}  (set ADASS_LOAD_MODEL=1 for the GPU sections)")
RESULTS = {}

# `adass.save_results` MERGES at the top level, so no later cell can delete a section it did not
# compute -- a response to a real incident: on 21 August a fresh process ran only the cheap
# sections, hit the first save with an empty RESULTS, and wiped hours of GPU output off disk.
# Rotating here means a run still starts clean, but only where a human can see it happen.
_out = adass.results_path(OUT)
if _out.exists():
    _prev = _out.with_suffix(".prev.json")
    _out.replace(_prev)
    print(f"rotated {_out.name} -> {_prev.name}")

In [ ]:
# %% 0.2 Module provenance.
#
# `adass/core.py` is now the SOURCE of the module. It used to be generated by a %%writefile cell
# in notebook 04, which is exactly how it silently drifted: on 23 August the file carried the
# repaired judge-v2 prompts while that cell still held v1, so re-running the cell would have
# reverted the judge repair and the negative control it exists to pass. The generation step is
# gone. What is worth asserting now is that the judge prompts still hash to the version the
# stored results were produced under -- otherwise cached judge output is stale by construction.
print("adass      ", adass.__version__, "from", Path(adass.__file__).parent)
print("judge hash ", adass.judge_prompt_hash())
_stored = json.load(open(adass.artifact("steps123_results.json")))["step3"]["prompt_hash"]
print("stored hash", _stored,
      "-- MATCH" if adass.judge_prompt_hash() == _stored else "-- CHANGED: recompute, do not reload")

In [ ]:
# %% 0.3 Environment check. float16 is a STOP condition: Gemma-2 emits broken text in fp16, and
# that failure is visually identical to the degeneration this project studies.
import torch

DEV, DT = adass.pick_device(), adass.pick_dtype(adass.pick_device())
print("device", DEV, "| dtype", DT)
assert DT is not torch.float16, "float16: STOP. See README, environment check."

CONFIG = json.load(open(adass.paths.config()))
SPL = adass.make_splits(seed=CONFIG["seed"])       # train_n MUST stay at its default 128 --
PROMPTS = SPL["harmless_test"]                     # train_n=160 shifts harmless_test by 32 items
assert len(PROMPTS) == 48
DIRS = torch.load(adass.artifact("refusal_dirs.pt"))
print(f"{len(PROMPTS)} test prompts | refusal_dirs {tuple(DIRS.shape)} | NORM_REF {adass.NORM_REF:.3f}")

MODEL = TOK = TO_CHAT = None
if LOAD_MODEL:
    MODEL, TOK, DT, DEV = adass.load_model()
    TO_CHAT = adass.make_chat_fn(TOK)
    print("model loaded")

## §1 The scorers

Two axes, each from the instrument that earned it against the 160 hand labels
(`docs/HANDOVER.md`, "Instruments, and which to trust for what"):

- **coherence** — the mechanical detector (2-of-3 votes over gzip compression, duplicate 5-grams,
  longest repeated span). It passes **both** controls unconditionally. The judge's coherence
  question does not: it clears its negative control but calls only 72.9% of known loops broken.
- **answered** — the binary judge, 91.4% against hand labels.
- **clean refusal** = coherent **and** not answered.

Thresholds are fitted on the two anchor conditions, never on the gold labels: `no-steer` is
coherent by construction and `dense/all m=2` is ~98% degenerate.

**Carried caveat, and it is load-bearing for everything below.** All 160 gold labels come from
*layer-16* conditions, so both instruments are validated on layer-16 text and applied here to
layer-10-to-20 text. That is out-of-distribution use. It is the reason the plan puts a ~40-item
blind label pass at a coherent layer immediately after this notebook, and no number below should
be quoted as hand-confirmed until that runs.

In [ ]:
# %% 1.1 Fit the mechanical thresholds on the anchors, and define the two axes.
GENS = json.load(open(adass.artifact("week3_generations.json")))
FIT = adass.fit_coherence_thresholds(GENS["no-steer"], GENS["dense/all m=2"])
for feat, d in FIT.items():
    print(f"  {feat:12} thr={d['threshold']:8.3f}  bacc={d['balanced_acc']:.3f}  margin={d['margin']:+.3f}")

def mech_broken(texts):
    return [adass.classify_mechanical(t, FIT)["broken"] for t in texts]

def judge_answered(prompts, texts):
    """None when the model is not loaded, so every section still completes."""
    if not LOAD_MODEL:
        return None
    out = adass.local_judge_binary(MODEL, TOK, TO_CHAT, list(zip(prompts, texts)))
    return [o["answered"] for o in out]

def score(texts, prompts=None, answered=None):
    """The row every table in this notebook reports. Wilson CIs on all three rates."""
    prompts = PROMPTS if prompts is None else prompts
    br = mech_broken(texts)
    ans = judge_answered(prompts, texts) if answered is None else answered
    n = len(texts)
    row = dict(n=n, broken=sum(br) / n, broken_ci=list(adass.wilson_ci(sum(br), n)),
               matcher=adass.refusal_rate(texts))
    if ans is not None:
        clean = [(not b) and (not a) for b, a in zip(br, ans)]
        row.update(suppressed=1 - sum(ans) / n,
                   suppressed_ci=list(adass.wilson_ci(n - sum(ans), n)),
                   clean_refusal=sum(clean) / n,
                   clean_refusal_ci=list(adass.wilson_ci(sum(clean), n)),
                   judge_answered=ans)
    row["mech_broken"] = br
    return row

def fmt(row, label=""):
    s = f"{label:26} broken {row['broken']:6.1%}"
    if "clean_refusal" in row:
        s += f" | suppressed {row['suppressed']:6.1%} | CLEAN REFUSAL {row['clean_refusal']:6.1%}"
    return s + f" | matcher {row['matcher']:6.1%}"

## §2 What the matcher could see — the selection was blind by construction

Notebook 01, cell 10:

```python
(BEST_LAYER, BEST_MULT) = max(sweep, key=sweep.get)   # sweep values are refusal_rate()
```

`refusal_rate` is the substring matcher later measured at **5.6% precision**. The operating point
was chosen by its argmax, and the re-selection was "lowest KL among **saturating** configs" — also
matcher-defined.

This section asks what that criterion could actually distinguish, by running it over the layer-sweep
generations. **No model needed.** The claim to test, stated before looking: if the matcher separates
the coherent regime from the broken one, it will fall away from 100% somewhere between layer 12 and
layer 20, tracking `clean_refusal`.

In [ ]:
# %% 2.1 The matcher across the layer sweep, against the two axes it was standing in for.
S123 = json.load(open(adass.artifact("steps123_results.json")))
SWEEP_GENS = {int(k): v for k, v in S123["step1"]["gens"].items()}
SWEEP_ROWS = {r["layer"]: r for r in S123["step1"]["rows"]}
LAYERS = sorted(SWEEP_GENS)

sat = []
for L in LAYERS:
    g, r = SWEEP_GENS[L], SWEEP_ROWS[L]
    k = int(round(adass.refusal_rate(g) * len(g)))
    sat.append(dict(layer=L, vec_norm=float(DIRS[L + 1].norm()),
                    matcher=adass.refusal_rate(g), matcher_ci=list(adass.wilson_ci(k, len(g))),
                    broken=r["broken"], suppressed=r["suppressed"],
                    clean_refusal=r["clean_refusal"]))

print(f"{'layer':>5} {'||v||':>8} {'MATCHER':>9} {'broken':>8} {'suppressed':>11} {'clean refusal':>14}")
for r in sat:
    print(f"{r['layer']:>5} {r['vec_norm']:>8.1f} {r['matcher']:>9.1%} {r['broken']:>8.1%} "
          f"{r['suppressed']:>11.1%} {r['clean_refusal']:>14.1%}")

full = [r["layer"] for r in sat if r["matcher"] == 1.0]
lo, hi = min(full), max(full)
span = [r for r in sat if lo <= r["layer"] <= hi]
drop = max(r["clean_refusal"] for r in span) - min(r["clean_refusal"] for r in span)
print(f"\nmatcher == 100% for layers {lo}-{hi}; over that span clean refusal moves {drop:.1%}")
print("A selection criterion constant over the range it selects cannot choose within it.")
RESULTS["s2_matcher_saturation"] = dict(rows=sat, saturated_span=[lo, hi],
                                        clean_refusal_range=drop)
print(adass.save_results(RESULTS, OUT))

In [ ]:
# %% 2.2 The report figure. Matcher flat; the quantity it stands in for collapsing underneath it.
import matplotlib.pyplot as plt

xs = [r["layer"] for r in sat]
fig, ax = plt.subplots(figsize=(6.4, 4.1))
ax.plot(xs, [r["matcher"] for r in sat], "o-", lw=2.4, label="substring matcher (weeks 1-2 metric)")
ax.plot(xs, [r["clean_refusal"] for r in sat], "s-", lw=2.0,
        label="clean refusal (coherent & not answered)")
ax.plot(xs, [r["broken"] for r in sat], "^--", lw=1.6, label="broken (mechanical)")
ax.axvspan(lo - 0.35, hi + 0.35, color="0.90", zorder=0)
ax.annotate("matcher saturated:\nno signal to select on", xy=(12.0, 0.52),
            ha="center", fontsize=8.5, color="0.30")
ax.axvline(16, color="crimson", ls=":", lw=1.4)
ax.annotate("layer 16 selected here,\nby this metric's argmax", xy=(16, 0.48), xytext=(16.7, 0.66),
            fontsize=8.5, color="crimson",
            arrowprops=dict(arrowstyle="->", color="crimson", lw=1.1))
ax.set_xlabel("steering layer"); ax.set_ylabel("rate over 48 harmless prompts")
ax.set_ylim(-0.04, 1.10); ax.set_xticks(xs)
ax.legend(fontsize=7.8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.20), frameon=False)
ax.set_title("The selection metric is flat where the behaviour changes", fontsize=10)
fig.tight_layout()
_fig = adass.figure("fig_matcher_saturation.png")
fig.savefig(_fig, dpi=200, bbox_inches="tight")
print("wrote", _fig.relative_to(adass.ROOT))

## §3 Reproduce the layer sweep

`steps123_results.json` has no code behind it in the repository. This regenerates step 1 —
`dense/all`, raw multiplier 1.0, 48 harmless test prompts, 128 tokens, greedy — and §3.2 diffs the
result against the stored run. Generation is greedy, so a matching environment should reproduce it
closely; systematic divergence means the environment differs (`transformers` major version, dtype,
padding side), and that is worth knowing before anything downstream is believed.

In [ ]:
# %% 3.1 The sweep. Raw multiplier 1.0 at each layer -- deliberately the CONFOUNDED
# parameterisation, because the job here is to reproduce what was run, not to improve it. §6 fixes it.
SWEEP_LAYERS = [8, 10, 12, 14, 16, 18, 20]
MAXNEW = 128

if LOAD_MODEL:
    rep = {}
    for L in SWEEP_LAYERS:
        v = DIRS[L + 1]                                   # INDEXED [layer + 1]
        g = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, layer=L, vector=v, mult=1.0,
                           positions="all", max_new_tokens=MAXNEW, device=DEV, dtype=DT)
        r = score(g)
        r.update(adass.strength_row(v, 1.0))
        rep[L] = dict(row=r, gens=g)
        print(fmt(r, f"layer {L}"))
        adass.empty_cache(DEV)
    RESULTS["s3_layer_sweep"] = {str(L): rep[L] for L in rep}
    print(adass.save_results(RESULTS, OUT))
else:
    rep = None
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 3.2 Diff against the 22 August run. Reports agreement, not equality: a greedy decode can
# still differ if the environment does, and the point is to find that out loudly.
if rep:
    print(f"{'layer':>5} {'broken 22Aug':>13} {'broken now':>11} {'clean 22Aug':>12} "
          f"{'clean now':>10} {'text match':>11}")
    diffs = {}
    for L in SWEEP_LAYERS:
        old, new = SWEEP_ROWS[L], rep[L]["row"]
        same = sum(a.strip() == b.strip() for a, b in zip(SWEEP_GENS[L], rep[L]["gens"])) / len(PROMPTS)
        diffs[L] = dict(broken_delta=new["broken"] - old["broken"],
                        clean_delta=new.get("clean_refusal", float("nan")) - old["clean_refusal"],
                        text_match=same)
        print(f"{L:>5} {old['broken']:>13.1%} {new['broken']:>11.1%} {old['clean_refusal']:>12.1%} "
              f"{new.get('clean_refusal', float('nan')):>10.1%} {same:>11.1%}")
    worst = max(abs(d["clean_delta"]) for d in diffs.values())
    print(f"\nlargest clean-refusal divergence: {worst:.1%}",
          "-- REPRODUCED" if worst <= 0.10 else "-- DIVERGENT: stop and diagnose the environment")
    RESULTS["s3_repro_diff"] = {str(L): diffs[L] for L in diffs}
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred")

## §4 The selection screen

Arditi et al.'s vector-selection procedure, as `docs/RELATED_WORK.md` §6 recommends adopting:
minimise `sigmoid(bypass) - sigmoid(induce)` subject to `induce > 0`, `kl < 0.1`, `layer < 0.8L`.
Reproduces step 2.

Read the KL column with the caveat the stored run recorded: **this KL is not Arditi's.** Theirs is
directional ablation on a harmless validation set; ours is `KL(steered || base)` on a fixed
reference under activation addition. Same spirit, different quantity — do not read the 0.1 threshold
as theirs. What the column is good for is the *ordering*, and the fact that our operating point sits
an order of magnitude above the threshold the field applies.

In [ ]:
# %% 4.1 Norm, KL, induce per layer.
if LOAD_MODEL:
    REF_TEXTS = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, max_new_tokens=MAXNEW,
                               device=DEV, dtype=DT)                       # unsteered reference
    base_m = adass.refusal_margin(MODEL, TOK, TO_CHAT, PROMPTS, device=DEV, dtype=DT)
    n_layers = MODEL.config.num_hidden_layers
    sel = []
    for L in SWEEP_LAYERS:
        v = DIRS[L + 1]
        kl = adass.kl_vs_base(MODEL, TOK, TO_CHAT, PROMPTS, REF_TEXTS, L, v, 1.0,
                              device=DEV, dtype=DT)
        induce = adass.refusal_margin(MODEL, TOK, TO_CHAT, PROMPTS, layer=L, vector=v, mult=1.0,
                                      device=DEV, dtype=DT)
        sel.append(dict(layer=L, norm=float(v.norm()), kl=kl, induce=induce,
                        induce_delta=induce - base_m,
                        passes_kl=bool(kl < 0.1), passes_induce=bool(induce > 0),
                        passes_layer=bool(L < 0.8 * n_layers)))
        print(f"layer {L:>2}  ||v||={sel[-1]['norm']:7.1f}  kl={kl:6.3f}  induce={induce:+.3f}  "
              f"kl_ok={sel[-1]['passes_kl']}")
    RESULTS["s4_selection"] = dict(rows=sel, base_margin=base_m, note=(
        "kl is KL(steered||base) on a fixed reference under ACTIVATION ADDITION; Arditi's kl score "
        "is directional ABLATION on harmless val. Do not read the 0.1 threshold as theirs."))
    print(adass.save_results(RESULTS, OUT))
    print("\nlayers passing the KL constraint:", [r["layer"] for r in sel if r["passes_kl"]] or "NONE")
else:
    print("deferred: needs ADASS_LOAD_MODEL=1")

## §5 The repaired judge's negative control

Blocking, and pre-registered in `docs/PLAN_REFUSAL_SUPPRESSION.md` phase 0 **before** the repair was
written:

> The negative control passes if the **Wilson 95% upper bound** on the false-broken rate is below
> **0.15**.

On n=48 that admits 0, 1 or 2 false-broken and fails at 3. The stored run passes at 1/48
(upper 0.109). This re-runs it. If `judge_prompt_hash()` has moved — §0.2 checks — the cached
numbers are stale by construction and must not be compared.

In [ ]:
# %% 5.1 Unsteered replies must not be called broken.
if LOAD_MODEL:
    jb = adass.local_judge_binary(MODEL, TOK, TO_CHAT, list(zip(PROMPTS, REF_TEXTS)))
    k = sum(o["broken"] for o in jb)
    ci = adass.wilson_ci(k, len(jb))
    ok = ci[1] < 0.15
    print(f"false-broken {k}/{len(jb)} = {k/len(jb):.1%}  Wilson95 upper {ci[1]:.3f}  "
          f"bar <0.15  -> {'PASS' if ok else 'FAIL'}")
    if not ok:
        print("BLOCKING: every 'answered' number in §3, §6 and §7 rests on this judge.")
    RESULTS["s5_judge_negative_control"] = dict(
        prompt_hash=adass.judge_prompt_hash(), false_broken=k, n=len(jb),
        ci=list(ci), bar="wilson_upper<0.15", pass_=bool(ok),
        which=[i for i, o in enumerate(jb) if o["broken"]])
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: needs ADASS_LOAD_MODEL=1")

## §6 The matched-norm test — pre-registered

**The confound.** `multiplier` scales the raw vector; `||v||` runs 48.7 → 303.6 over the sweep. So
"multiplier 1.0 at layer 10" is a perturbation **0.359x** the size of "multiplier 1.0 at layer 16".
Layer and strength moved together in every comparison this project has made, including the one that
produced the reversal. Two readings fit the same data and they imply different papers:

- *strength* — pushing 2.8x too hard breaks the model, and layer 16 is unremarkable;
- *depth* — layer 16 is a genuinely worse site to intervene at, at any magnitude.

**The design.** Two ladders, each holding the perturbation norm fixed while depth moves:

- **Ladder A** — layer 16 at the raw multipliers that make `||m·v_16||` equal `||v_L||` for
  L ∈ {8, 10, 12, 14, 16}: m ∈ {0.282, 0.359, 0.551, 0.709, 1.000}.
- **Ladder B** — the converse: layers 10, 12, 14 pushed **up** to `||v_16|| = 172.54`:
  m ∈ {2.786, 1.817, 1.410}.

n=48 harmless test prompts, 128 tokens, greedy, `positions="all"`. Coherence from the mechanical
detector, answering from the judge, exactly as §1 defines them.

**The decision rule, fixed before the run.** Written here so that whichever way it lands, the
reading was not chosen afterwards:

| outcome | reading |
|---|---|
| Ladder A at `‖v_10‖` reaches clean refusal ≥ 80% **and** broken ≤ 10% | **Strength, not depth.** The layer finding is a strength artefact; the coherent regime is a function of perturbation magnitude and layer 16 was never the problem |
| Ladder A at `‖v_10‖` stays broken > 30% **and** Ladder B at `‖v_16‖` stays broken ≤ 10% | **Depth, not strength.** Layer 16 is genuinely a worse intervention site at matched magnitude |
| both ladders track perturbation norm, and layer shifts clean refusal by < 15 points at matched norm | **Strength dominates, depth is second-order.** The relative axis in §7 becomes primary and layer becomes a nuisance parameter |
| anything else | the surface is genuinely two-dimensional; report it as such and do not reduce it to one factor |

**What does not depend on the outcome.** §2 stands either way: the matcher is flat across layers
10–18 whichever factor drives the behaviour underneath it, so the selection procedure was blind
regardless. Both answers are reportable; neither rescues nor destroys the headline.

In [ ]:
# %% 6.1 Both ladders.
TARGETS = {L: float(DIRS[L + 1].norm()) for L in SWEEP_LAYERS}

LADDER_A = [(16, TARGETS[L], f"L16 @ ||v_{L}||") for L in [8, 10, 12, 14, 16]]
LADDER_B = [(L, TARGETS[16], f"L{L} @ ||v_16||") for L in [10, 12, 14]]

if LOAD_MODEL:
    matched = {}
    for layer, target, label in LADDER_A + LADDER_B:
        v = DIRS[layer + 1]
        m = adass.mult_matching_norm(v, target)
        g = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, layer=layer, vector=v, mult=m,
                           positions="all", max_new_tokens=MAXNEW, device=DEV, dtype=DT)
        r = score(g)
        r.update(adass.strength_row(v, m), label=label, layer=layer, target_norm=target)
        matched[label] = dict(row=r, gens=g)
        print(fmt(r, f"{label} (m={m:.3f})"))
        adass.empty_cache(DEV)
    RESULTS["s6_matched_norm"] = matched
    print(adass.save_results(RESULTS, OUT))
else:
    matched = None
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 6.2 Apply the pre-registered rule. Printed as a verdict, not as a paragraph to interpret later.
if matched:
    a10 = matched["L16 @ ||v_10||"]["row"]
    b_rows = [matched[f"L{L} @ ||v_16||"]["row"] for L in [10, 12, 14]]

    strength = a10["clean_refusal"] >= 0.80 and a10["broken"] <= 0.10
    depth    = a10["broken"] > 0.30 and all(r["broken"] <= 0.10 for r in b_rows)

    # everything sitting at ||v_16||: how much does depth ALONE move the outcome?
    at_ref = b_rows + [matched["L16 @ ||v_16||"]["row"]]
    spread = max(r["clean_refusal"] for r in at_ref) - min(r["clean_refusal"] for r in at_ref)
    dominates = (not strength) and (not depth) and spread < 0.15

    verdict = ("STRENGTH, NOT DEPTH" if strength else
               "DEPTH, NOT STRENGTH" if depth else
               "STRENGTH DOMINATES, DEPTH SECOND-ORDER" if dominates else
               "TWO-DIMENSIONAL -- report the surface")
    print(f"L16 @ ||v_10||   : broken {a10['broken']:.1%}  clean refusal {a10['clean_refusal']:.1%}")
    print(f"clean-refusal spread across layers at matched norm ||v_16||: {spread:.1%}")
    print(f"\nVERDICT: {verdict}")
    RESULTS["s6_verdict"] = dict(verdict=verdict, strength=bool(strength), depth=bool(depth),
                                 dominates=bool(dominates), layer_spread_at_ref=spread)
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred")

## §7 The relative axis

Absolute matching under-corrects: the residual stream itself grows with depth, so a fixed
perturbation norm is a progressively *smaller* relative perturbation the deeper it lands. The
dimensionless quantity is `||m·v|| / ||h||` at that layer.

The ladder is anchored to the operating point rather than guessed, so it is guaranteed to bracket
the regime the project has been running in: measure `r16 = ||v_16|| / ||h_16||`, then sweep
`r ∈ r16 · {0.25, 0.5, 1.0, 1.5}` at layers 10, 12, 14, 16. Sixteen cells, n=48 each.

This is the axis every later comparison — H1, H2, H3 — should use. Ranking masking schemes at
matched *raw* multiplier is what made week 2's H1 test unfair; matched raw multiplier across layers
is the same mistake one dimension over.

In [ ]:
# %% 7.1 Relative-strength grid.
GRID_LAYERS = [10, 12, 14, 16]
REL_FACTORS = [0.25, 0.5, 1.0, 1.5]

if LOAD_MODEL:
    HN = {L: adass.mean_hidden_norm(MODEL, TOK, TO_CHAT, PROMPTS, L, device=DEV)
          for L in GRID_LAYERS}
    r16 = float(DIRS[17].norm()) / HN[16]
    print("mean ||h|| per layer:", {L: round(v, 1) for L, v in HN.items()})
    print(f"relative strength at the weeks 1-3.5 operating point (L16, m=1): {r16:.4f}\n")

    grid = {}
    for L in GRID_LAYERS:
        v = DIRS[L + 1]
        for f in REL_FACTORS:
            rel = r16 * f
            m = adass.rel_mult_for(v, rel, HN[L])
            g = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, layer=L, vector=v, mult=m,
                               positions="all", max_new_tokens=MAXNEW, device=DEV, dtype=DT)
            r = score(g)
            r.update(adass.strength_row(v, m, HN[L]), layer=L, rel_factor=f)
            grid[f"L{L}/rel{f}"] = dict(row=r, gens=g)
            print(fmt(r, f"L{L} rel={rel:.4f} (x{f})"))
            adass.empty_cache(DEV)
    RESULTS["s7_relative_grid"] = dict(h_norms={str(k): v for k, v in HN.items()},
                                       r16=r16, cells=grid)
    print(adass.save_results(RESULTS, OUT))
else:
    grid = None
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 7.2 The surface: does relative strength collapse the layer differences?
if grid:
    print(f"{'rel x':>7} " + " ".join(f"{'L' + str(L):>17}" for L in GRID_LAYERS))
    print(f"{'':>7} " + " ".join(f"{'clean / broken':>17}" for _ in GRID_LAYERS))
    for f in REL_FACTORS:
        cells = [grid[f"L{L}/rel{f}"]["row"] for L in GRID_LAYERS]
        print(f"{f:>7.2f} " + " ".join(f"{c['clean_refusal']:>8.1%} /{c['broken']:>7.1%}"
                                       for c in cells))
    spreads = {f: max(grid[f"L{L}/rel{f}"]["row"]["clean_refusal"] for L in GRID_LAYERS)
                - min(grid[f"L{L}/rel{f}"]["row"]["clean_refusal"] for L in GRID_LAYERS)
               for f in REL_FACTORS}
    print("\nclean-refusal spread across layers, at matched RELATIVE strength:")
    for f, s in spreads.items():
        print(f"  rel x{f:<5} {s:.1%}")
    print("\nSmall spreads => depth was a proxy for strength all along and the relative axis "
          "removes it.\nLarge spreads => depth is real and survives normalisation. Either is a result.")
    RESULTS["s7_layer_spread"] = {str(k): v for k, v in spreads.items()}
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred")

## §8 What this notebook settled

Filled in from the run, not in advance. The three things it is meant to leave behind:

1. a reproducible layer sweep, so the reversal has code behind it;
2. a verdict on the strength/depth confound under a rule written before the numbers;
3. a normalised strength axis for H1/H2/H3, so the next comparison cannot repeat the mistake.

In [ ]:
# %% 8.1 Summary.
print("=" * 78)
for key, label in [("s2_matcher_saturation", "matcher saturated over layers"),
                   ("s3_repro_diff", "layer sweep reproduced"),
                   ("s5_judge_negative_control", "judge negative control"),
                   ("s6_verdict", "matched-norm verdict"),
                   ("s7_layer_spread", "layer spread at matched relative strength")]:
    r = RESULTS.get(key)
    if r is None:
        print(f"{label:44} -- not run")
    elif key == "s2_matcher_saturation":
        print(f"{label:44} layers {r['saturated_span'][0]}-{r['saturated_span'][1]}, "
              f"clean refusal moves {r['clean_refusal_range']:.1%} underneath")
    elif key == "s3_repro_diff":
        w = max(abs(d["clean_delta"]) for d in r.values())
        print(f"{label:44} largest divergence {w:.1%}")
    elif key == "s5_judge_negative_control":
        print(f"{label:44} {r['false_broken']}/{r['n']}, {'PASS' if r['pass_'] else 'FAIL'}")
    elif key == "s6_verdict":
        print(f"{label:44} {r['verdict']}")
    else:
        print(f"{label:44} " + ", ".join(f"x{k}: {v:.1%}" for k, v in r.items()))
print("=" * 78)
print(adass.save_results(RESULTS, OUT))